In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import matplotlib as mpl
sys.path.append('../.')

# Import only what's actually needed to avoid the dask import issue
from GrindingData import GrindingData
from MyDataset import project_dir
from MyModels import GrindingPredictor
from GrindingData import GrindingData
from MyDataset import project_dir, allowed_input_types, get_dataset
from plot_time_series_simple import load_physics_data, find_bdi_indices, load_trained_model, generate_predictions_for_model, plot_time_series_base, plot_time_series_simple


# # Define the functions that were imported from plot_prediction_simple
def load_physics_data():
    """
    Load physics data including surface roughness and BDI values.
    """
    # Create GrindingData instance
    grinding_data = GrindingData(project_dir)
    
    # Load only physics data (much more efficient)
    print("Loading physics data...")
    grinding_data._load_all_physics_data()
    
    # Extract the data we need
    true_values = grinding_data.sr * 1e3  # Convert to μm
    bdi_values = grinding_data.bid
    st_values = grinding_data.st
    
    # Convert to numpy arrays and ensure proper shape
    true_values = np.array(true_values).flatten()
    bdi_values = np.array(bdi_values).flatten()
    st_values = np.array(st_values).flatten()
    
    print(f"Loaded physics data for {len(true_values)} samples")
    print(f"BDI range: {np.min(bdi_values):.3f} to {np.max(bdi_values):.3f}")
    print(f"Surface roughness range: {np.min(true_values):.3f} to {np.max(true_values):.3f} μm")
    
    return true_values, bdi_values, st_values

def find_bdi_indices(bdi_values, threshold=1.0):
    """
    Find indices where BDI transitions between brittle and ductile regimes.
    """
    bdi_regime = bdi_values > threshold
    regime_changes = np.where(np.diff(bdi_regime.astype(int)) != 0)[0] + 1
    regime_starts = np.concatenate(([0], regime_changes))
    regime_ends = np.concatenate((regime_changes, [len(bdi_regime)]))
    return regime_starts, regime_ends, bdi_regime

# Placeholder functions for the missing imports
def load_trained_model(model_path):
    """Placeholder for loading trained model"""
    print(f"Would load model from: {model_path}")
    return None

def generate_predictions_for_model(model, data):
    """Placeholder for generating predictions"""
    print("Would generate predictions")
    return None

def plot_time_series_base(true_values, predictions, bdi_values, title="Prediction vs Ground Truth"):
    """
    Create a simple time series plot with BDI regime coloring.
    """
    sample_indices = np.arange(len(true_values))
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot ground truth and predictions
    ax.plot(sample_indices, true_values, 'o-', label='Ground Truth', 
            color='black', alpha=0.8, markersize=4, linewidth=1.5)
    
    if predictions is not None:
        ax.plot(sample_indices, predictions, 's-', label='Prediction', 
                color='red', alpha=0.8, markersize=4, linewidth=1.5)
    
    # Color background based on BDI regime
    regime_starts, regime_ends, bdi_regime = find_bdi_indices(bdi_values)
    
    for start, end in zip(regime_starts, regime_ends):
        regime = bdi_regime[start]
        color = 'lightblue' if regime else 'lightcoral'
        alpha = 0.3 if regime else 0.2
        
        # Use integer indices for sample positions
        x_start = sample_indices[max(0, start)]
        x_end = sample_indices[min(len(sample_indices)-1, end)]
        
        ax.axvspan(x_start, x_end, ymin=0, ymax=1, alpha=alpha, color=color)
    
    # Customize plot
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Surface Roughness Ra (um)')
    ax.set_title(title)
    
    # Create legend with regime information
    legend_elements = [
        Line2D([0], [0], color='black', marker='o', linestyle='-', label='Ground Truth'),
        Patch(facecolor='lightblue', alpha=0.3, label='BDI > 1 (Ductile)'),
        Patch(facecolor='lightcoral', alpha=0.2, label='BDI < 1 (Brittle)')
    ]
    
    if predictions is not None:
        legend_elements.insert(1, Line2D([0], [0], color='red', marker='s', linestyle='-', label='Prediction'))
    
    ax.legend(handles=legend_elements, loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig, ax

In [2]:
def load_trained_model(model_type="ae_features", fold=0):
    """
    Load a trained model from lfs/checkpoints directory.
    """
    
    # Construct model path relative to current working directory
    model_filename = f"{model_type}_fold{fold}_of_folds10.pt"
    model_path = os.path.join("../lfs/checkpoints", model_filename)
    
    if not os.path.exists(model_path):
        print(f"Model file not found: {model_path}")
        # Check what models are available
        checkpoints_dir = "../lfs/checkpoints"
        if os.path.exists(checkpoints_dir):
            available_models = [f for f in os.listdir(checkpoints_dir) if f.endswith('.pt')]
            print(f"Available models: {available_models[:10]}...")  # Show first 10
        else:
            print(f"Checkpoints directory not found: {checkpoints_dir}")
        return None
    
    # Initialize model with correct input type
    model = GrindingPredictor(input_type=model_type)
    
    # Load model weights
    try:
        checkpoint = torch.load(model_path, map_location='cpu')
        if 'model_state' in checkpoint:
            model.load_state_dict(checkpoint['model_state'])
        elif 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            model.load_state_dict(checkpoint)
        print(f"Loaded model: {model_filename}")
    except Exception as e:
        print(f"Error loading model: {e}")
        return None
    
    model.eval()
    return model

def generate_predictions_for_all_models():
    """
    Generate predictions for all model types using their respective checkpoints.
    Returns a dictionary with predictions for each model type.
    """
    
    predictions_dict = {}
    
    for model_type in allowed_input_types:
        print(f"\n=== Generating predictions for {model_type} ===")
        
        # Load model
        model = load_trained_model(model_type, fold=0)
        if model is None:
            print(f"Failed to load model for {model_type}")
            continue
        
        # Load dataset based on model type
        try:
            dataset = get_dataset(input_type=model_type, dataset_mode="classical")
        except Exception as e:
            print(f"Error loading dataset for {model_type}: {e}")
            continue
        
        predictions = []
        true_values = []
        bdi_values = []
        
        # Generate predictions for each sample
        print(f"Generating predictions using {model_type} model...")
        with torch.no_grad():
            for i in range(min(100, len(dataset))):  # Use first 100 samples for efficiency
                try:
                    item = dataset[i]
                    
                    # Prepare input batch
                    batch = {}
                    for key in ['spec_ae', 'spec_vib', 'features_ae', 'features_vib', 'features_pp']:
                        if key in item:
                            batch[key] = item[key].unsqueeze(0)  # Add batch dimension
                    
                    # Add lengths if available
                    if 'features_ae' in item:
                        batch['ae_lengths'] = torch.tensor([item['features_ae'].shape[0]])
                    if 'features_vib' in item:
                        batch['vib_lengths'] = torch.tensor([item['features_vib'].shape[0]])
                    
                    # Generate prediction
                    prediction = model(batch)
                    if isinstance(prediction, tuple):
                        prediction = prediction[0]  # Handle case where model returns (prediction, attention)
                    
                    predictions.append(prediction.item())
                    true_values.append(item['label'].item())
                    
                    # Extract BDI (3rd element in features_pp: [ec, st, bid])
                    if 'features_pp' in item:
                        bdi_values.append(item['features_pp'][2].item())
                    else:
                        # If no features_pp, use default BDI value
                        bdi_values.append(1.0)
                        
                except Exception as e:
                    print(f"Error processing sample {i} for {model_type}: {e}")
                    continue
        
        if len(predictions) > 0:
            predictions_dict[model_type] = {
                'true_values': np.array(true_values),
                'predictions': np.array(predictions),
                'bdi_values': np.array(bdi_values)
            }
            print(f"Generated {len(predictions)} predictions for {model_type}")
            print(f"MAE: {np.mean(np.abs(np.array(true_values) - np.array(predictions))):.3f} μm")
        else:
            print(f"No predictions generated for {model_type}")
    
    return predictions_dict

In [6]:
model_type=allowed_input_types[0]
model = load_trained_model
dataset = get_dataset(input_type=model_type, dataset_mode="classical")
model

<function __main__.load_trained_model(model_type='ae_features', fold=0)>

In [9]:
item = dataset[0]
item

{'label': tensor(112),
 'features_pp': tensor([0.7284, 0.0000, 0.0000]),
 'features_ae': tensor([[0.0413, 0.0378, 0.0134,  ..., 0.0065, 0.0226, 0.0190],
         [0.0390, 0.0307, 0.0167,  ..., 0.0084, 0.0486, 0.0395],
         [0.5556, 0.4444, 0.6667,  ..., 0.3333, 0.4444, 0.4444],
         [0.4615, 0.3846, 0.4615,  ..., 0.2308, 0.4615, 0.4615]]),
 'spec_ae': tensor([[[[-25.9288, -33.4342, -36.8104,  ..., -48.0844, -34.8646, -46.1319],
           [-28.1438, -40.2344, -27.2027,  ..., -19.7401, -20.0636, -31.5557],
           [-23.7362, -27.8281, -19.2636,  ..., -16.8388, -20.7722, -26.6928],
           ...,
           [-78.9217, -80.0000, -78.0530,  ..., -79.2395, -80.0000, -79.6818],
           [-80.0000, -80.0000, -80.0000,  ..., -80.0000, -80.0000, -77.8523],
           [-80.0000, -80.0000, -80.0000,  ..., -80.0000, -79.1219, -77.9345]],
 
          [[-16.4466, -42.7402, -22.1990,  ..., -46.2477, -28.9176, -24.1672],
           [-10.2349, -18.3327, -21.4609,  ..., -25.4187, -20.3929,

In [11]:
model

<function __main__.load_trained_model(model_type='ae_features', fold=0)>

In [2]:
folds=10
repeat=10
for _input_type in allowed_input_types[:-1]:
    for i in range(int(folds*repeat)):
        checkpoint_path = f"{_input_type}_fold{i}_of_folds{folds}.pt"
        checkpoint_path = os.path.join(CHECKPOINT_DIR, checkpoint_path)
# _input_type = allowed_input_types[0]
_input_type = 'all'
_input_type

NameError: name 'CHECKPOINT_DIR' is not defined

In [3]:
folds=10
repeat=10
checkpoint_path = f"{_input_type}_fold{0}_of_folds{folds}.pt"
checkpoint_path = os.path.join(CHECKPOINT_DIR, checkpoint_path)

dataset = get_dataset(input_type=_input_type)
collate_fn = get_collate_fn(input_type=_input_type)
loader = DataLoader(dataset, batch_size=3, shuffle=False, num_workers=0, collate_fn=collate_fn)
model = GrindingPredictor(input_type=_input_type, interp=False)
model.to(device)
checkpoint = torch.load(checkpoint_path, map_location=device)
_state_dict = checkpoint['model_state']
model.load_state_dict(_state_dict, strict=True)

Required components: {'all'}


<All keys matched successfully>

In [3]:
folds=10
repeat=10
for _input_type in allowed_input_types[:-1]:
    for i in range(int(folds*repeat))[:1]:
        checkpoint_path = f"{_input_type}_fold{i}_of_folds{folds}.pt"
        checkpoint_path = os.path.join(CHECKPOINT_DIR, checkpoint_path)

        dataset = get_dataset(input_type=_input_type)
        collate_fn = get_collate_fn(input_type=_input_type)
        loader = DataLoader(dataset, batch_size=3, shuffle=False, num_workers=0, collate_fn=collate_fn)
        model = GrindingPredictor(input_type=_input_type, interp=False)
        model.to(device)
        checkpoint = torch.load(checkpoint_path, map_location=device)
        _state_dict = checkpoint['model_state']
        model.load_state_dict(_state_dict, strict=True)

Required components: {'ae_spec'}
Required components: {'vib_spec'}
Required components: {'ae_features'}
Required components: {'vib_features'}
Required components: {'ae_spec', 'ae_features'}
Required components: {'vib_spec', 'vib_features'}
Required components: {'vib_spec', 'ae_spec', 'ae_features', 'vib_features'}
Required components: {'pp', 'ae_features'}
Required components: {'pp', 'vib_features'}
Required components: {'pp'}
Required components: {'vib_spec', 'ae_spec'}


FileNotFoundError: [Errno 2] No such file or directory: '../lfs/checkpoints/ae_spec+vib_spec_fold0_of_folds10.pt'

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device)
_state_dict = checkpoint['model_state']
model.load_state_dict(_state_dict, strict=False)

RuntimeError: Error(s) in loading state_dict for GrindingPredictor:
	size mismatch for regressor.0.weight: copying a param with shape torch.Size([128, 64]) from checkpoint, the shape in current model is torch.Size([128, 192]).

In [8]:
batch = next(iter(loader))
inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
labels = batch['label'].to(device)
pred = model.forward(inputs)
pred, labels

(tensor([[34.3949],
         [33.2866],
         [33.1813]], grad_fn=<AddmmBackward0>),
 tensor([112, 112, 105]))

(tensor([[34.3949],
         [33.2866],
         [33.1813]], grad_fn=<AddmmBackward0>),
 tensor([112, 112, 105]))

In [17]:
checkpoint['model_state']

OrderedDict([('ae_spec_processor.conv.0.weight',
              tensor([[[[ 1.8810e-01,  9.7475e-02, -4.0856e-02],
                        [ 2.0492e-01,  1.7017e-01,  1.7988e-02],
                        [ 4.0255e-02, -1.3628e-01, -3.5494e-01]],
              
                       [[-1.0835e-01,  7.8401e-02,  8.1748e-02],
                        [ 9.1444e-02, -8.1134e-02, -1.8812e-01],
                        [-1.9436e-01,  4.1149e-02, -1.1562e-01]]],
              
              
                      [[[-1.6569e-01,  1.3392e-01, -1.8564e-01],
                        [-5.6104e-02, -1.9228e-01,  1.2668e-01],
                        [ 2.2337e-01, -7.4634e-02, -1.5330e-01]],
              
                       [[-1.3344e-01, -2.6960e-01, -1.5580e-01],
                        [ 3.0235e-01, -1.5618e-01,  1.6963e-01],
                        [-2.3885e-01, -1.2401e-02,  2.3923e-01]]],
              
              
                      [[[ 1.1516e-01,  1.0318e-01, -1.0890e-01],
          